# SACDpy batch single-image reconstruction

This notebook batch-processes single-channel processed TIFF videos into one SACD image per input. Edit the settings cell, preview the planned filenames, then run the batch cell.

## 1. Setup

Run this notebook from the SACDpy package folder. The local `src/` folder is added to the Python path when the package is not installed.

In [1]:
from pathlib import Path
import sys

repo = Path.cwd()
src = repo / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.batch import find_input_files, run_batch_reconstruction, sacdpy_output_path

print(f"Working folder: {repo}")

Working folder: /Users/gmgao/GGscripts/GG-general-GuttmanLab/image_analysis/pipelines/SACDpy


## 2. Settings

These are the only values most users need to edit. With `output_dir=None`, output TIFFs are written next to the input TIFFs.

In [2]:
input_path = Path("/Volumes/guttman/users/gmgao/Imaging_ProcessedData/Collaborations/20260625_ONI-gmgao-SPRITE_SACD-DAPI_RL560")
glob_pattern = "*.tif"
exclude_name_contains = ("SACDpy",)
output_dir = None
overwrite_outputs = False
show_progress = True

pixel_nm = 117.0
na = 1.45
default_wavelength_nm = 561
wavelength_by_name = {"DAPI": 450, "RL560": 580}

mag = 2
iter1 = 7
iter2 = 8
ac_order = 2
subfactor = 0.8
frames_per_sacd = None  # one SACD image from each full processed video

# Optional SACDm branches. Defaults reproduce the validated SACDm baseline.
ifbackground = False
backgroundfactor = 2.0
ifregistration = False
ifsparsedecon = False
fidelity = 100.0
tcontinuity = 0.1
sparsity = 1.0
sparse_iterations = 100

## 3. Preview inputs and outputs

Run this cell first to confirm the notebook found the expected files and will write the expected output names.

In [3]:
input_files = find_input_files(input_path, glob_pattern, exclude_name_contains=exclude_name_contains)
if not input_files:
    raise FileNotFoundError(f"No input TIFF files found from {input_path} with pattern {glob_pattern!r}")

print(f"Found {len(input_files)} input TIFF(s):")
for input_file in input_files:
    output_file = sacdpy_output_path(input_file, output_dir=output_dir)
    print(f"{input_file.name} -> {output_file.name}")

Found 21 input TIFF(s):
SPRITE_clusters_SACD-DAPI-noRL560-FOV-1-DAPI.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-1-DAPI-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-1-RL560.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-1-RL560-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-2-DAPI.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-2-DAPI-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-2-RL560.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-2-RL560-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-3-DAPI.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-3-DAPI-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-3-RL560.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-3-RL560-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-4-DAPI.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-4-DAPI-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-4-RL560.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-4-RL560-SACDpy.tif
SPRITE_clusters_SACD-DAPI-noRL560-FOV-DAPI.tif -> SPRITE_clusters_SACD-DAPI-noRL560-FOV-DAPI-SACDpy.tif


## 4. Run batch reconstruction

This cell writes float32 SACD TIFFs. Existing outputs are skipped unless `overwrite_outputs=True`.

In [4]:
results = run_batch_reconstruction(
    input_files,
    output_dir=output_dir,
    overwrite_outputs=overwrite_outputs,
    pixel_nm=pixel_nm,
    na=na,
    default_wavelength_nm=default_wavelength_nm,
    wavelength_by_name=wavelength_by_name,
    mag=mag,
    iter1=iter1,
    iter2=iter2,
    ac_order=ac_order,
    subfactor=subfactor,
    frames_per_sacd=frames_per_sacd,
    ifbackground=ifbackground,
    backgroundfactor=backgroundfactor,
    ifregistration=ifregistration,
    ifsparsedecon=ifsparsedecon,
    fidelity=fidelity,
    tcontinuity=tcontinuity,
    sparsity=sparsity,
    sparse_iterations=sparse_iterations,
    show_progress=show_progress,
)

for result in results:
    if result.status == "written":
        print(f"Wrote {result.output.name} | {result.input_shape} -> {result.output_shape} | {result.wavelength_nm:g} nm")
    else:
        print(f"Skipped {result.output.name} ({result.status})")

results

Output()

ValueError: could not broadcast input array from shape (153,153,0) into shape (153,153,2)